# 使用 Unsloth 对 DeepSeek-R1-Distill-Qwen-1.5B 模型进行 LoRA 微调

本 Notebook 展示了如何使用 `unsloth` 库对 `deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B` 模型进行高效的 QLoRA (Low-Rank Adaptation) 微调。

整个流程包括：
1.  环境准备与库导入
2.  加载预训练模型和分词器 (Tokenizer)。
3.  在微调前，对模型进行简单的推理测试。
4.  下载和格式化训练数据集
5.  使用 `unsloth` 的 `FastLanguageModel` 来为模型添加 LoRA 适配器。
6.  配置 `SFTTrainer` 监督微调训练配置。
7.  启动训练，并观察 Loss 变化情况
8.  保存微调后的模型
9.  测试训练后的生成结果

### 1. 环境准备与库导入

首先，我们需要安装并导入所有必要的库。`transformers` 用于加载模型和分词器，`unsloth` 用于高效微调，`trl` 提供了 `SFTTrainer`，而 `datasets` 用于处理数据。

**注意**: 在运行此 Notebook 之前，请确保已安装所有依赖包：
```bash
pip install -r requirements.txt
```

In [1]:
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, GenerationConfig, DataCollatorForSeq2Seq
from datasets import Dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/miniconda3/envs/distill/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


🦥 Unsloth Zoo will now patch everything to make training faster!


### 2. 加载预训练模型和分词器 (Tokenizer)

In [2]:
# 定义模型和一些基本参数
max_seq_length = 8192
dtype = None # None 表示自动选择 (Float16 a T4, V100, BFloat16 a Ampere)
load_in_4bit = True # 使用 4bit 量化加载

# 这是您的模型标识符，请替换为您正在使用的模型
# 例如："qwen-1.5b_lora_model"
# model_name = "qwen-1.5b_lora_model" 
# model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B" 
model_name = "unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit" 

# 这一步会返回一个经过 Unsloth 优化的模型和一个分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

==((====))==  Unsloth 2025.8.5: Fast Qwen2 patching. Transformers: 4.55.2.
   \\   /|    NVIDIA GeForce RTX 3090. Num GPUs = 1. Max memory: 23.569 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.7.1+cu126. CUDA: 8.6. CUDA Toolkit: 12.6. Triton: 3.3.1
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.31.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/1.81G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/472 [00:00<?, ?B/s]

### 3. 微调前推理测试

在对模型进行任何修改之前，我们先用它来生成一段文本，看看原始模型的表现如何。这可以作为我们微调效果的基准参考。

In [3]:
# 模型推理的 Prompt 模板
inference_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}
"""

In [4]:
FastLanguageModel.for_inference(model)

question = "男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。"

inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")
attention_mask = inputs.input_ids.ne(tokenizer.pad_token_id).long().to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)

In [5]:
response = tokenizer.batch_decode(outputs, skip_special_tokens=True)

In [6]:
print(response[0].split("### Response:")[1])


<think>
好的，我现在要处理这个医学问题。让我先仔细阅读一下问题描述。

这位28岁的男性，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。首先，我需要考虑他的症状，头晕、脖子疼、恶心，这些都是常见的症状，但结合他的职业和工作时间，可能需要更深入的分析。

首先，头晕可能是由于脑力劳动过度导致的，比如长时间工作可能会影响脑供血不足，导致头晕。脖子疼可能与工作相关，比如长时间使用电脑、手机等，也可能与颈椎问题有关。恶心可能与长期使用药物、咖啡因或酒精有关，或者与长期的压力或焦虑有关。

接下来，我需要考虑他的职业和工作环境。程序员通常工作在信息处理、编程等岗位，工作环境可能与日常办公不同，比如长时间使用键盘、电脑，可能带来疲劳。工作时间到半夜，可能意味着他有其他工作，如学习、休息，或者可能有加班的情况。

现在，我需要构建一个分步的思考过程：

1. **症状分析**：首先，明确症状是头晕、脖子疼、恶心，以及持续的时间。这些症状可能与长期的工作压力或疲劳相关。

2. **职业背景**：程序员通常工作在信息处理领域，工作时间较长，可能影响脑供血和神经功能。工作环境可能与日常办公不同，比如可能有加班或学习，导致疲劳。

3. ** possible causes**：考虑可能的原因，如长期使用电子设备、长时间工作导致的疲劳，颈椎问题，或与长期压力相关。

4. **可能的诊断**：可能需要检查脑供血不足、颈椎病、压力或焦虑等，也可能需要考虑药物治疗，如抗抑郁药或抗焦虑药，如果出现恶心。

5. **后续评估**：如果诊断为脑供血不足，可能需要进行脑供血灌注（CBN）或其他相关治疗。如果脖子疼或颈椎问题，可能需要进一步检查和治疗。

6. **综合治疗**：综合考虑，可能需要药物治疗和相关康复治疗，以缓解症状。

7. **建议**：建议咨询医生，评估具体情况，可能需要进一步的检查和治疗。

现在，我需要确保回答逻辑严谨，内容准确，并且符合医学知识的范畴。
</think>

男，28岁，程序员，最近一周每天工作到半夜，感觉头晕、脖子疼，有时候还恶心。

首先，考虑他的症状：头晕、脖子疼、恶心，可能与长期工作压力、疲劳、颈椎问题等有关。由于他工作时间到半夜，可能涉及加班或学习，带来持续的疲劳。

可能的诊断包括：
1. **脑供血不足**：头晕可能

---

### 4. 下载和格式化训练数据集


医学推理数据集：https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/viewer/zh

![dataset](images/dataset.png)

In [6]:
# 模型训练的 Prompt 模板
train_prompt = """以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>
{}
</think>
{}
"""

In [7]:
EOS_TOKEN = tokenizer.eos_token # 添加 EOS Token

def formatting_prompts_func(examples):
    inputs = examples["Question"]
    cots = examples["Complex_CoT"]
    outputs = examples["Response"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        # 将 EOS Token 添加到样本最后
        text = train_prompt.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }
pass

from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/medical-o1-reasoning-SFT", "zh", split = "train")
dataset = dataset.map(formatting_prompts_func, batched = True,)

README.md: 0.00B [00:00, ?B/s]

medical_o1_sft_Chinese.json:   0%|          | 0.00/50.6M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/20171 [00:00<?, ? examples/s]

In [9]:
dataset[0]["text"]

'以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。\n请撰写一份恰当的回复，以完成该请求。\n在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。\n\n\n### Instruction:\n你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。\n请回答以下医学问题。\n\n### Question:\n根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？\n\n### Response:\n<think>\n这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。\n\n用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。\n\n但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。\n\n等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。\n\n仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。\n\n回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？\n\n再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。\n\n看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。\n\n不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。\n\n

In [10]:
from IPython.display import display, Markdown

display(Markdown(dataset[0]["text"]))

以下是一条描述任务的指令，并配有一个提供进一步上下文的输入。
请撰写一份恰当的回复，以完成该请求。
在回答之前，请仔细思考该问题，并构建一个分步的思考过程，以确保回应的逻辑严谨和内容准确。


### Instruction:
你是一位医学专家，在临床推理、诊断学和治疗规划方面拥有深厚的专业知识。
请回答以下医学问题。

### Question:
根据描述，一个1岁的孩子在夏季头皮出现多处小结节，长期不愈合，且现在疮大如梅，溃破流脓，口不收敛，头皮下有空洞，患处皮肤增厚。这种病症在中医中诊断为什么病？

### Response:
<think>
这个小孩子在夏天头皮上长了些小结节，一直都没好，后来变成了脓包，流了好多脓。想想夏天那么热，可能和湿热有关。才一岁的小孩，免疫力本来就不强，夏天的湿热没准就侵袭了身体。

用中医的角度来看，出现小结节、再加上长期不愈合，这些症状让我想到了头疮。小孩子最容易得这些皮肤病，主要因为湿热在体表郁结。

但再看看，头皮下还有空洞，这可能不止是简单的头疮。看起来病情挺严重的，也许是脓肿没治好。这样的情况中医中有时候叫做禿疮或者湿疮，也可能是另一种情况。

等一下，头皮上的空洞和皮肤增厚更像是疾病已经深入到头皮下，这是不是说明有可能是流注或瘰疬？这些名字常描述头部或颈部的严重感染，特别是有化脓不愈合，又形成通道或空洞的情况。

仔细想想，我怎么感觉这些症状更贴近瘰疬的表现？尤其考虑到孩子的年纪和夏天发生的季节性因素，湿热可能是主因，但可能也有火毒或者痰湿造成的滞留。

回到基本的症状描述上看，这种长期不愈合又复杂的状况，如果结合中医更偏重的病名，是不是有可能是涉及更深层次的感染？

再考虑一下，这应该不是单纯的瘰疬，得仔细分析头皮增厚并出现空洞这样的严重症状。中医里头，这样的表现可能更符合‘蚀疮’或‘头疽’。这些病名通常描述头部严重感染后的溃烂和组织坏死。

看看季节和孩子的体质，夏天又湿又热，外邪很容易侵入头部，对孩子这么弱的免疫系统简直就是挑战。头疽这个病名听起来真是切合，因为它描述的感染严重，溃烂到出现空洞。

不过，仔细琢磨后发现，还有个病名似乎更为合适，叫做‘蝼蛄疖’，这病在中医里专指像这种严重感染并伴有深部空洞的情况。它也涵盖了化脓和皮肤增厚这些症状。

哦，该不会是夏季湿热，导致湿毒入侵，孩子的体质不能御，其病情发展成这样的感染？综合分析后我觉得‘蝼蛄疖’这个病名真是相当符合。
</think>
从中医的角度来看，你所描述的症状符合“蝼蛄疖”的病症。这种病症通常发生在头皮，表现为多处结节，溃破流脓，形成空洞，患处皮肤增厚且长期不愈合。湿热较重的夏季更容易导致这种病症的发展，特别是在免疫力较弱的儿童身上。建议结合中医的清热解毒、祛湿消肿的治疗方法进行处理，并配合专业的医疗建议进行详细诊断和治疗。
<｜end▁of▁sentence｜>

### 5. 使用 Unsloth 添加 LoRA 适配器

这是使用 `unsloth` 的核心步骤。我们调用 `FastLanguageModel.get_peft_model`，它会非常高效地为模型注入 LoRA 模块。

- `r`: LoRA 的秩 (rank)，是控制模型复杂度和参数量的关键超参数。
- `target_modules`: 指定要在哪些线性层（如注意力机制的 q, k, v, o 投影层）上应用 LoRA。
- `lora_alpha`: LoRA 的缩放因子，通常设置为 `r` 的两倍或与 `r` 相同。
- `use_gradient_checkpointing`: 一种节省显存的技术，对于训练大模型至关重要。

In [11]:
# 因为 `model` 对象现在是由 Unsloth 创建的，它包含了所有必需的属性
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
      "q_proj",
      "k_proj",
      "v_proj",
      "o_proj",
      "gate_proj",
      "up_proj",
      "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=1432,
    use_rslora=False,
    loftq_config=None,
)
# 检查模型结构，确认 LoRA 适配器已添加
print(model)

Unsloth 2025.8.5 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): Embedding(151936, 1536, padding_idx=151654)
        (layers): ModuleList(
          (0): Qwen2DecoderLayer(
            (self_attn): Qwen2Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1536, out_features=1536, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1536, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=1536, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
       

### 6. 配置 SFTTrainer

`SFTTrainer` (Supervised Fine-tuning Trainer) 是一个专门用于指令微调的训练器。我们需要配置 `TrainingArguments` 来指定所有的训练参数，如批量大小、学习率、优化器等。

In [12]:
from trl import SFTConfig, SFTTrainer
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    packing = False, # Can make training 5x faster for short sequences.
    args = SFTConfig(
        per_device_train_batch_size = 64,
        gradient_accumulation_steps = 2,
        warmup_steps = 5,
        # num_train_epochs = 1, # Set this for 1 full training run.
        max_steps = 60,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1432,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/20171 [00:00<?, ? examples/s]

### 7. 开始训练

一切准备就绪后，调用 `trainer.train()` 即可开始微调过程。训练结束后，会返回包含训练统计信息（如训练损失）的对象。

In [13]:
trainer_stats = trainer.train()

# 打印训练统计信息
print(trainer_stats)

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 20,171 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 64 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (64 x 2 x 1) = 128
 "-____-"     Trainable parameters = 18,464,768 of 1,795,552,768 (1.03% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,3.160700
2,3.169100
3,3.078300
4,3.167400
5,3.150900
6,3.008300
7,2.984500
8,2.946800
9,2.900600
10,2.794600


'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0f4432ed-1a91-4de9-bec3-6ca2021b1fce)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: ef0f6d00-94f5-4096-b48e-97b815d13312)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 7dacab2a-1dd4-4e04-aa15-58505d752b57)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'(ReadTimeoutError("HT

TrainOutput(global_step=60, training_loss=2.479310937722524, metrics={'train_runtime': 1347.64, 'train_samples_per_second': 5.699, 'train_steps_per_second': 0.045, 'total_flos': 7.163728311484416e+16, 'train_loss': 2.479310937722524})


### 8. 保存微调后的模型（Lora）

训练完成后，您可以再次进行推理，比较微调后的模型与原始模型的差异。如果对结果满意，可以使用 `model.save_pretrained("your_lora_adapter_path")` 来保存训练好的 LoRA 适配器。

In [14]:
model.save_pretrained("qwen-1.5b_lora_model")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 5c1bd913-9e41-4c5b-a90b-3645472bd886)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 13e027af-9354-4c29-a283-867a7f83886d)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 1237b8ed-1afa-491b-b23d-a73e4ad3d7d7)')' thrown while requesting HEAD https://huggingface.co/unsloth/DeepSeek-R1-Distill-Qwen-1.5B-unsloth-bnb-4bit/resolve/main/config.json
Retrying in 4s [Retry 3/5].
'(ReadTimeoutError("HT

In [15]:
tokenizer.save_pretrained("qwen-1.5b_lora_model")

('qwen-1.5b_lora_model/tokenizer_config.json',
 'qwen-1.5b_lora_model/special_tokens_map.json',
 'qwen-1.5b_lora_model/chat_template.jinja',
 'qwen-1.5b_lora_model/tokenizer.json')

In [20]:
# 模型保存方式二选一（要么使用上面的分开保存，要么使用这里的合并 Lora 保存）
# model.save_pretrained_merged("qwen-1.5b_lora_model", tokenizer, save_method="merged_16bit")

### 9. 测试训练后的生成结果

In [16]:
FastLanguageModel.for_inference(model) # Enable native 2x faster inference

question="一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？", # Question
inputs = tokenizer([inference_prompt.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1000,
    use_cache=True,
)

In [17]:
output = tokenizer.batch_decode(outputs, skip_special_tokens=True)
print(output[0].split("### Response:")[1])


<think>
这位病人，5天已经得了急性阑尾炎，还有一点腹痛和发热，但还是有右下腹的压痛包块。这个压痛包块，看起来很严重，可能就是阑尾炎的典型表现。

首先，这个压痛包块是阑尾炎的常见症状之一，它通常来自阑尾部的炎症反应，尤其是急性阶段。所以，根据这个情况，需要立即处理。

急性阑尾炎的患者应该立即就医，因为症状已经很严重，这会带来严重的后果。因此，立即进行紧急处理是必要的。

立即的紧急处理包括立即就医，这能帮助医生快速诊断和治疗。然而，患者在紧急情况下可能需要立即停止用药，尤其是抗生素，以避免进一步的感染。

根据这个情况，患者应该立即停止抗生素治疗。这可能有助于减轻感染，防止进一步扩散。同时，观察压痛包块是否还有其他症状，比如疼痛或肿胀，如果症状没有改善，可能需要进一步的治疗。

在紧急情况下，及时处理是非常重要的。因此，患者需要立即就医，并立即停止抗生素使用。

综上所述，立即就医是治疗急性阑尾炎的必要步骤。患者应该立即停止抗生素，并观察症状，以便获得进一步的诊断和治疗。
</think>
根据患者的症状描述，急性阑尾炎的患者已经发病5天，腹痛稍有减轻但仍然发热，并且在体检时发现右下腹有压痛的包块。在这种情况下，立即的紧急处理是必要的。

急性阑尾炎通常表现为压痛包块，这通常与阑尾炎的急性阶段有关。因此，患者需要立即就医以获得准确的诊断和治疗。

为了防止进一步的感染，患者应该立即停止使用抗生素。由于压痛包块仍然存在，建议立即观察症状，以便获得进一步的诊断和治疗建议。

因此，患者应立即就医，并立即停止抗生素治疗以避免进一步的感染风险。



In [18]:
def generate_response(question: str, model, tokenizer, inference_prompt: str, max_new_tokens: int = 1024) -> str:
    """
    使用指定的模型和分词器为给定的医学问题生成响应。

    Args:
        question (str): 需要模型回答的医学问题。
        model: 已加载的 Unsloth/Hugging Face 模型。
        tokenizer: 对应的分词器。
        inference_prompt (str): 用于格式化输入的 f-string 模板。
        max_new_tokens (int, optional): 生成响应的最大 token 数量。默认为 1024。

    Returns:
        str: 模型生成的响应文本，已去除 prompt 部分。
    """
    # 1. 使用模板格式化输入
    prompt = inference_prompt.format(
        question, # 填充问题
        "",       # 留空，让模型生成 CoT 和 Response
    )

    # 2. 将格式化后的 prompt 进行分词，并转移到 GPU
    inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    # 3. 使用模型生成输出
    # use_cache=True 用于加速解码过程
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=max_new_tokens,
        use_cache=True,
    )
    
    # 4. 将生成的 token 解码为文本
    # skip_special_tokens=True 会移除像 EOS_TOKEN 这样的特殊标记
    decoded_output = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]

    # 5. 切分字符串，只返回 "### Response:" 之后的部分
    # 使用 .split() 分割并获取响应内容，.strip() 用于去除可能存在的前后空白字符
    response_part = decoded_output.split("### Response:")
    if len(response_part) > 1:
        return response_part[1].strip()
    else:
        # 如果模型没有生成 "### Response:" 标记，则返回整个生成内容以供调试
        return decoded_output

In [19]:
my_question = "对于一名60岁男性患者，出现右侧胸疼并在X线检查中显示右侧肋膈角消失，诊断为肺结核伴右侧胸腔积液，请问哪一项实验室检查对了解胸水的性质更有帮助？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
嗯，这位60岁的男性患者出现右侧胸疼，X线检查显示右侧肋膈角消失。这让我想到可能有肺结核。可是，根据X线检查的结果，右侧胸腔里好像有积液。这让我有点困惑，因为肺结核通常会导致胸腔积液，但通常在急性肺结核时，积液是由于肺部的病变，比如肺泡结节，导致的。

不过，这次看起来，积液可能是由于肺部的其他病变，比如肺结的结节或者其他结构的病变，而不是肺部的肺泡结节。这让我有点困惑，因为肺结核的诊断通常需要通过胸腔积液来判断。

所以，我想到，如果要了解这个胸腔积液的性质，可能需要一些辅助检查。比如，可能需要做CT扫描，因为CT可以提供更清晰的影像，帮助我们了解胸腔的结构和液体的性质。

此外，我还需要考虑一下肺结核的其他症状，比如发热、咳嗽、呼吸困难等，这些症状可能也会影响CT结果。所以，CT可能是最直接的检查方法，因为它可以帮助我们了解液体的性质，比如是否是液体、是否是气体，以及液体的体积。

综上所述，CT扫描应该是最能帮助我们了解这个胸腔积液性质的检查，因为它能提供详细的信息，帮助我们判断液体的性质，从而更好地诊断肺结核。
</think>
对于这位60岁男性患者，由于X线检查显示右侧胸腔里有积液，结合他的症状，可能提示他患有肺结核。为了更好地了解胸腔积液的性质，可以考虑使用CT扫描作为辅助检查。CT扫描可以提供更清晰的影像，帮助我们评估胸腔的结构和液体的性质，比如液体还是气体，以及液体的体积。因此，CT扫描是评估胸腔积液性质的最佳选择。


In [20]:
my_question = "对于一名 28 岁的男性患者，工作是程序员，常年熬夜，最近突然感觉头晕目眩，甚至有点恶心。请问有可能是什么疾病？"

response = generate_response(my_question, model, tokenizer, inference_prompt)
print("==================== 模型回答 ====================")
print(response)

==================== 模型回答 ====================
<think>
这位28岁的男性患者，工作是程序员，长期熬夜，最近突然出现头晕目眩和恶心的症状，这让我想到他最近可能有脑损伤。不过，也有可能是因为睡眠不足导致的脑损伤。还有一种可能是他长期处于高压环境，比如工作压力大，导致脑部损伤。

不过，我还需要考虑他最近有没有出现过其他症状，比如恶心、呕吐这些，这可能与脑损伤的类型有关。如果他最近没有出现这些症状，那么可能是因为睡眠不足导致的脑损伤。

现在，我需要回顾一下脑损伤的分类，看看这些症状是否符合其中一种分类。常见的脑损伤包括脑缺血、脑缺氧、脑缺血性脑坏、脑缺氧性脑坏、脑缺氧性脑损伤和脑缺血性脑损伤。

根据他的症状，头晕目眩和恶心，这些症状都与脑损伤有关，尤其是与脑缺血性脑损伤或脑缺氧性脑损伤有关。

那么，可能的脑损伤类型包括脑缺血性脑损伤、脑缺氧性脑损伤，以及脑缺血性脑坏。

接下来，我需要考虑这些可能的脑损伤类型。比如，脑缺血性脑损伤可能是由于脑缺血导致的，而脑缺氧性脑损伤则是由于脑缺氧导致的。

根据他最近出现的症状，这些可能是由长期高压环境引起的，因为他在程序员的工作中长期处于高压环境，这种高压可能影响了他的神经系统。

所以，脑缺血性脑损伤和脑缺氧性脑损伤都可能符合他的症状。

总结一下，根据他最近的症状和工作环境，脑缺血性脑损伤和脑缺氧性脑损伤都可能是可能的。

因此，综合来看，最有可能的是脑缺血性脑损伤或者脑缺氧性脑损伤。这两种疾病都可能导致头晕目眩和恶心，尤其是长期高压环境导致的脑损伤。

所以，综合考虑，我倾向于认为他可能有脑缺血性脑损伤，因为这种疾病可能与长期高压环境和脑损伤有关。
</think>
根据患者的情况，这位28岁的男性患者工作是程序员，长期熬夜，最近突然出现头晕目眩和恶心症状，这些症状提示他可能遭受脑损伤。根据脑损伤的分类，常见的脑损伤包括脑缺血性脑损伤、脑缺氧性脑损伤、脑缺血性脑坏和脑缺氧性脑坏。考虑到患者长期处于高压环境，脑缺血性脑损伤和脑缺氧性脑损伤都可能符合他的症状。

脑缺血性脑损伤通常与脑缺血有关，而脑缺氧性脑损伤则与脑缺氧有关。这两种疾病都可能导致头晕目眩和恶心，尤其是长期高压环境导致的脑损伤。

因此，综合考虑，最有可能的是脑缺血性脑损伤或脑缺氧性脑损伤。这两种疾病都可能